# AELIONIX BLACKFORGE — Phase 2 Colab Validation

This notebook performs a deterministic, one-click validation of the Blackforge **Persistent Memory** foundation (Phase 2).

**What this validates:**
- Repository integrity and commit verification
- Dependency installation (runtime + dev extras)
- All Blackforge imports, including the new memory modules
- Full automated test suite
- Bootstrap + health verification (`memory_ready`)
- Persistent memory write/close/reopen across process restarts
- Full memory record roundtrip: provenance, confidence, evidence refs, lifecycle
- Deterministic deduplication and logical versioning
- Structured search filters (mission, session, status, lifecycle, source, confidence, tags, keyword)
- Transaction atomicity (rollback on failure)

**What this does NOT do:**
- No model download or LLM inference (Phase 2 memory is stdlib-only)
- No offensive security actions, reconnaissance, or scanning
- No autonomous attack planning

**Runtime:** Google Colab (CPU or GPU) — the notebook runs identically on free CPU runtimes.

---
## 1. Runtime Information

In [ ]:
import sys
import platform

print("Blackforge Phase 2 Colab Validation (Persistent Memory)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---
## 2. Repository Acquisition

In [ ]:
from pathlib import Path
import subprocess

# ── Configuration (edit here if fork changes) ──────────────────────────
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# ───────────────────────────────────────────────────────────────────────

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

import os
os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---
## 3. Commit Verification

In [ ]:
import subprocess

EXPECTED_PHASE1_COMMIT = "1e54de4"

result = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h"],
    capture_output=True, text=True, check=True,
)
current_commit = result.stdout.strip()

print(f"Expected Phase 1 baseline: {EXPECTED_PHASE1_COMMIT}")
print(f"Current repository commit: {current_commit}")

result_log = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "--oneline"],
    capture_output=True, text=True, check=True,
)
commits = [line.split()[0] for line in result_log.stdout.strip().splitlines()]

# Phase 2 lives on top of the Phase 1 baseline. The memory modules must exist.
has_phase2_memory = (
    (REPO_DIR / "blackforge" / "memory" / "repository.py").exists()
    and (REPO_DIR / "blackforge" / "memory" / "manager.py").exists()
)

if not has_phase2_memory:
    raise RuntimeError("Phase 2 memory modules not present — repo is ahead or behind.")
print("Phase 2 memory modules present: PASS")

if EXPECTED_PHASE1_COMMIT in commits:
    print("Commit verification: PASS (Phase 1 baseline found)")
elif any(c.startswith(EXPECTED_PHASE1_COMMIT[:4]) for c in commits):
    print("Commit verification: PASS (Phase 1 baseline found, abbreviated match)")
else:
    result_merge = subprocess.run(
        ["git", "-C", str(REPO_DIR), "merge-base", "--is-ancestor",
         EXPECTED_PHASE1_COMMIT, current_commit],
        capture_output=True, check=False,
    )
    if result_merge.returncode == 0:
        print("Commit verification: PASS (repo advanced past Phase 1)")
    else:
        print(f"WARNING: Phase 1 commit {EXPECTED_PHASE1_COMMIT} not found in history.")
        print("The repo may predate Phase 1. Continuing anyway.")

---
## 4. Install Blackforge

Phase 2 memory uses only the Python standard library. Only the `[dev]` extra is installed — the `llm` extra (torch/transformers) is intentionally left out, so this notebook is lightweight and OOM-free on free CPU runtimes.

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]"

import blackforge
print("Blackforge import: PASS")


---
## 5. Environment / Import Health Check

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.core.logging",
    "blackforge.runtime.bootstrap",
    "blackforge.runtime.hardware",
    "blackforge.memory",
    "blackforge.memory.base",
    "blackforge.memory.provenance",
    "blackforge.memory.repository",
    "blackforge.memory.manager",
    "blackforge.memory.models",
    "blackforge.evidence.store",
    "blackforge.evidence.models",
    "blackforge.mission.manager",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.validator",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Memory module imports: PASS")

---
## 6. Automated Regression Tests

In [ ]:
import subprocess
import sys

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---
## 7. Bootstrap Verification

Bootstraps the app and confirms every subsystem — including `memory_ready` — reports healthy.

In [ ]:
from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
assert verification["memory_ready"], "memory_ready must be True"

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap: PASS")

---
## 8. Persistent Memory — Restart Survival

The core Phase 2 guarantee: memory written through one manager instance survives closing and reopening the database. We prove it by writing with one `SQLiteMemoryRepository` connection, fully closing it, then reopening a **brand new** connection to the same file.

In [ ]:
import time
from pathlib import Path

from blackforge.core.types import EvidenceID, EvidenceStatus, MissionID, SessionID
from blackforge.memory.base import MemoryLifecycle, MemoryRecord, MemoryType
from blackforge.memory.manager import MemoryManager
from blackforge.memory.provenance import MemoryProvenance, MemorySource
from blackforge.memory.repository import SQLiteMemoryRepository

DB_PATH = str(Path("data/phase2_validation.db"))
Path("data").mkdir(exist_ok=True)

# Fresh database for a deterministic run (safe on re-runs)
for stale in [Path(DB_PATH), Path("data/phase2_search.db"), Path("data/phase2_txn_validation.db")]:
    stale.unlink(missing_ok=True)

# ── Writer instance ───────────────────────────────────────────────────────
m1 = MemoryManager(SQLiteMemoryRepository(DB_PATH))

rec = MemoryRecord(
    memory_type=MemoryType.KNOWLEDGE,
    key="survivor",
    content={"finding": "phase2_restart_proof", "ts": 1},
    status=EvidenceStatus.VALIDATED,
    confidence=0.95,
    mission_id=MissionID("mission_phase2"),
    session_id=SessionID("sess_phase2"),
    source=MemorySource.CAPABILITY_EXECUTION,
    provenance=MemoryProvenance(
        source=MemorySource.CAPABILITY_EXECUTION,
        source_detail="mock_discovery",
        output_hash="sha256:abc123",
    ),
    lifecycle=MemoryLifecycle.ACTIVE,
    tags=["validation", "restart"],
    metadata={"phase": 2},
    evidence_ids=[EvidenceID("ev_phase2_1")],
)

written_id = m1.store(rec)
print(f"Stored record id={written_id} key='survivor'")
print(f"count in writer: {m1.count()}")

# Fully close the writer (process/restart simulation)
m1.close()
print("Writer closed.")

# ── Reader instance (brand new connection) ───────────────────────────────
m2 = MemoryManager(SQLiteMemoryRepository(DB_PATH))
loaded = m2.retrieve(written_id)

assert loaded is not None, "Record did NOT survive restart"
assert loaded.key == "survivor"
assert loaded.content == {"finding": "phase2_restart_proof", "ts": 1}
assert loaded.status == EvidenceStatus.VALIDATED
assert loaded.confidence == 0.95
assert loaded.mission_id == MissionID("mission_phase2")
assert loaded.session_id == SessionID("sess_phase2")
assert loaded.provenance.output_hash == "sha256:abc123"
assert loaded.evidence_ids == [EvidenceID("ev_phase2_1")]
assert loaded.lifecycle == MemoryLifecycle.ACTIVE

print(f"Reloaded id={loaded.id} key={loaded.key} content={loaded.content}")
print(f"  status={loaded.status.value} confidence={loaded.confidence}")
print(f"  lifecycle={loaded.lifecycle.value} tags={loaded.tags}")
print(f"  provenance={loaded.provenance.source.value} output_hash={loaded.provenance.output_hash}")
print(f"  evidence_ids={[str(e) for e in loaded.evidence_ids]}")

print("Persistent memory restart survival: PASS")

---
## 9. Deduplication & Logical Versioning

Storing identical `(type, key, content)` is a no-op (returns the same ID, no duplicate rows). Changing content under the same logical key creates a new version and marks the previous one `SUPERSEDED` — atomically.

In [ ]:
# Continuing on the reopened manager (m2)

from blackforge.memory.base import MemoryQuery

# 1) Idempotent dedup
dup_id = m2.store(
    MemoryRecord(
        memory_type=MemoryType.KNOWLEDGE,
        key="survivor",
        content={"finding": "phase2_restart_proof", "ts": 1},
        mission_id=MissionID("mission_phase2"),
    )
)
assert dup_id == written_id, "Dedup failed: idempotent write returned a new ID"
assert m2.count() == 1, "Dedup failed: duplicate row created"
dedup_count = m2.count()
print(f"Dedup: identical write returned same id {dup_id}; count={m2.count()} — PASS")

# 2) Content change -> new version, old superseded
v2_id = m2.store(
    MemoryRecord(
        memory_type=MemoryType.KNOWLEDGE,
        key="survivor",
        content={"finding": "phase2_restart_proof", "ts": 2},
        mission_id=MissionID("mission_phase2"),
    )
)
assert v2_id != written_id, "Versioning failed: content change kept the same ID"

old = m2.retrieve(written_id)
new = m2.retrieve(v2_id)
assert old is not None and new is not None
assert old.lifecycle == MemoryLifecycle.SUPERSEDED, "Old version not superseded"
assert new.version == 2, f"Expected version 2, got {new.version}"
assert new.supersedes == old.id

current = m2.find_by_logical_key(MemoryType.KNOWLEDGE, "survivor")
assert current.id == new.id, "find_by_logical_key must return the newest version"

print(f"Versioning: v1 superseded, v2 created (id {v2_id}) — PASS")
print(f"  find_by_logical_key -> version {current.version}")

---
## 10. Structured Search Filters

MemoryQuery supports deterministic filtering by type, mission, session, status, lifecycle, source, confidence range, tags, keyword, and time window.

In [ ]:
from blackforge.memory.manager import MemoryManager
from blackforge.memory.repository import SQLiteMemoryRepository
from blackforge.memory.base import MemoryQuery

# Fresh dedicated database so the filter expectations are exact.
sm = MemoryManager(SQLiteMemoryRepository(str(Path("data/phase2_search.db"))))

seed = [
    MemoryRecord(memory_type=MemoryType.KNOWLEDGE, key="svc_nginx", content="nginx on 443",
                 status=EvidenceStatus.VALIDATED, confidence=0.9,
                 mission_id=MissionID("mission_a"), session_id=SessionID("sess_a"),
                 source=MemorySource.TOOL_OUTPUT, tags=["web"],
                 metadata={"port": 443}),
    MemoryRecord(memory_type=MemoryType.KNOWLEDGE, key="svc_ssh", content="ssh on 22",
                 status=EvidenceStatus.OBSERVED, confidence=0.5,
                 mission_id=MissionID("mission_a"), session_id=SessionID("sess_b"),
                 source=MemorySource.USER_PROVIDED, tags=["ssh"],
                 metadata={"port": 22}),
    MemoryRecord(memory_type=MemoryType.EXPERIENCE, key="lesson", content="rotate secrets",
                 status=EvidenceStatus.VALIDATED, confidence=0.7,
                 mission_id=MissionID("mission_b"), session_id=SessionID("sess_a"),
                 source=MemorySource.LLM_INFERENCE, tags=["ops"],
                 metadata={}),
]
for r in seed:
    sm.store(r)

checks = {
    "by type (knowledge)": MemoryQuery(memory_type=MemoryType.KNOWLEDGE),
    "by mission_a": MemoryQuery(mission_id=MissionID("mission_a")),
    "by session sess_a": MemoryQuery(session_id=SessionID("sess_a")),
    "validated status": MemoryQuery(status=EvidenceStatus.VALIDATED),
    "active lifecycle": MemoryQuery(lifecycle=MemoryLifecycle.ACTIVE),
    "tool_output source": MemoryQuery(source=MemorySource.TOOL_OUTPUT),
    "confidence >= 0.7": MemoryQuery(confidence_min=0.7),
    "confidence <= 0.5": MemoryQuery(confidence_max=0.5),
    "tag web": MemoryQuery(tags=["web"]),
    "keyword nginx": MemoryQuery(keyword="nginx"),
}

expect = [2, 2, 2, 2, 3, 1, 2, 1, 1, 1]

for (label, q), expected in zip(checks.items(), expect):
    n = len(sm.search(query=q))
    ok = n == expected
    print(f"  [{'PASS' if ok else 'FAIL'}] {label:<20} -> {n} (expected {expected})")
    assert ok, f"{label}: expected {expected}, got {n}"

print("Structured search filters: PASS")

---
## 11. Transaction Atomicity

If any statement inside a transaction fails, the entire transaction rolls back — no partial writes are persisted.

In [ ]:
import sqlite3

txn_db = SQLiteMemoryRepository(str(Path("data/phase2_txn_validation.db")))
before = txn_db.count()

partial = MemoryRecord(
    memory_type=MemoryType.EXPERIENCE,
    key="should_not_exist",
    content="partial write",
)

try:
    with txn_db.transaction():
        txn_db.store(partial)
        raise sqlite3.OperationalError("simulated mid-transaction failure")
except sqlite3.OperationalError:
    print("Transaction raised (expected), rollback executed.")

after = txn_db.count()
print(f"count before: {before}, count after rollback: {after}")
assert after == before, "Transaction did not roll back cleanly"
assert txn_db.retrieve(partial.id) is None, "Partial record survived rollback"
txn_db.close()

print("Transaction atomicity: PASS")

---
## 12. Evidence Integration

Memory references evidence by ID (no copies). The evidence store keeps the canonical record; memory keeps a typed `evidence_ids` reference plus the shared content hash in provenance.

In [ ]:
from blackforge.core.types import EvidenceID, EvidenceStatus, EvidenceType, Confidence
from blackforge.evidence.models import Evidence, Provenance as EvidenceProvenance
from blackforge.evidence.store import EvidenceStore

ev_store = EvidenceStore()
evidence = ev_store.add(
    Evidence(
        mission_id=MissionID("mission_phase2"),
        source_capability="mock_discovery",
        target="example.com",
        evidence_type=EvidenceType.OBSERVATION,
        status=EvidenceStatus.VALIDATED,
        confidence=Confidence.HIGH,
        raw_data="nginx detected on port 443",
        provenance=EvidenceProvenance(
            capability_id="mock_discovery",
            output_hash="sha256:abc123",
        ),
    )
)

mem = MemoryRecord(
    memory_type=MemoryType.EVIDENCE,
    key="evidence-summary",
    content={"finding": "nginx on 443", "evidence_ref": str(evidence.id)},
    status=EvidenceStatus.VALIDATED,
    confidence=0.9,
    mission_id=MissionID("mission_phase2"),
    source=MemorySource.CAPABILITY_EXECUTION,
    provenance=MemoryProvenance(
        source=MemorySource.CAPABILITY_EXECUTION,
        source_detail="mock_discovery",
        output_hash="sha256:abc123",
    ),
    evidence_ids=[evidence.id],
)
m2.store(mem)

mem_id = m2.store(mem)
stored = m2.retrieve(mem_id)
assert stored is not None
assert stored.evidence_ids == [evidence.id], "Evidence reference not preserved"
assert stored.provenance.output_hash == evidence.provenance.output_hash

canonical = ev_store.get(evidence.id)
assert canonical is not None
assert canonical.raw_data == "nginx detected on port 443"
assert ev_store.count() == 1

print(f"Evidence id:      {evidence.id}")
print(f"Memory ref:       {stored.evidence_ids[0]}")
print(f"Output hash match: {stored.provenance.output_hash} == {canonical.provenance.output_hash}")
print("Evidence integration (reference, not copy): PASS")

---
## 13. In-Memory Backend Parity

The `in_memory` configuration (used in tests) behaves identically for store/dedupe/version/query semantics.

In [ ]:
from blackforge.memory.repository import InMemoryRepository

im = MemoryManager(InMemoryRepository())
i1 = im.store(MemoryRecord(memory_type=MemoryType.WORKING, key="w", content="x"))
i2 = im.store(MemoryRecord(memory_type=MemoryType.WORKING, key="w", content="x"))
assert i1 == i2, "In-memory dedup failed"
assert im.count() == 1
assert im.health_check() is True

print(f"In-memory dedup: same id {i1}, count={im.count()} — PASS")
print("In-memory backend parity: PASS")

---
## 14. PASS/FAIL Summary

In [ ]:
RESULTS = {}

RESULTS["repository"] = (REPO_DIR / "blackforge" / "__init__.py").exists()
RESULTS["phase2_modules"] = (
    (REPO_DIR / "blackforge" / "memory" / "repository.py").exists()
    and (REPO_DIR / "blackforge" / "memory" / "manager.py").exists()
)
RESULTS["imports"] = len(_import_failures) == 0
RESULTS["tests"] = True  # Would have raised if failed
RESULTS["bootstrap"] = app.healthy() and verification["memory_ready"]
RESULTS["restart_persistence"] = (
    loaded is not None and loaded.key == "survivor"
)
RESULTS["dedup"] = dup_id == written_id and dedup_count == 1
RESULTS["versioning"] = v2_id != written_id and new.version == 2
RESULTS["search_filters"] = all(
    len(sm.search(query=q)) == e
    for q, e in [(qr, x) for (_, qr), x in zip(checks.items(), expect)]
)
RESULTS["transaction_atomicity"] = after == before
RESULTS["evidence_integration"] = (
    stored.evidence_ids == [evidence.id]
    and stored.provenance.output_hash == canonical.provenance.output_hash
)
RESULTS["in_memory_parity"] = i1 == i2 and im.count() == 1

print("=" * 60)
print("BLACKFORGE PHASE 2 VALIDATION (PERSISTENT MEMORY)")
print("=" * 60)

labels = {
    "repository": "Repository",
    "phase2_modules": "Phase 2 modules",
    "imports": "Imports",
    "tests": "Automated tests",
    "bootstrap": "Bootstrap + memory_ready",
    "restart_persistence": "Restart persistence",
    "dedup": "Deterministic dedup",
    "versioning": "Logical versioning",
    "search_filters": "Structured search",
    "transaction_atomicity": "Transaction atomicity",
    "evidence_integration": "Evidence integration",
    "in_memory_parity": "In-memory parity",
}

for key in labels:
    status = "PASS" if RESULTS[key] else "FAIL"
    print(f"{labels[key]:<28} {status}")

print("=" * 60)

if all(RESULTS.values()):
    print("OVERALL RESULT: PASS")
else:
    failed = [labels[k] for k, v in RESULTS.items() if not v]
    print(f"OVERALL RESULT: FAIL — {', '.join(failed)}")

print("=" * 60)

In [ ]:
# Clean up: close all memory managers
m1.close()
m2.close()
print("Memory managers closed. Validation complete.")